# 03a — Shared Survival-Modeling Dataset (`v1`)

One narrow, versioned modeling dataset that Person 2 (feature-combination analysis), Person 3
(alternative survival models), and Person 4 (centralized evaluation + NACC harmonization) can all
load — so nobody re-joins raw ADNI data.

## Built ONLY from the frozen primary cohort
Input: `outputs/01c_mci_survival_cohort_freeze/mci_survival_primary_cohort_v1.tsv` (Build 1, v1).
This notebook **does not**: re-join raw tables · re-anchor · redefine MCI eligibility · recompute
event/censor dates · exclude short-follow-up or prior-dementia participants · impute · scale ·
select features · add cognition · fit models · generate CV folds.

## Design
- **All 401 participants preserved.** Optional biomarkers stay **NA where missing** — the dataset is
  NOT reduced to any complete-case subset.
- Raw biomarker values carried byte-for-byte; the only derived variables are `APOE4_CARRIER`,
  `log_ptau217`, `log_gfap`, `log_nfl` (matching Person 2's published analysis), the already-frozen
  `ratio_ab42_ab40`, and 8 per-participant `eligible_*` analysis-set flags.
- Two prior-dementia QC fields are carried through so downstream sensitivity analyses need no re-join.

**Exactly 26 columns, in a fixed order (asserted).**

## 0. Setup, read-only guards, and input checksum verification

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


def find_project_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "Data" / "raw").is_dir():
            return cand
    raise FileNotFoundError("Could not locate Data/raw above %s" % start)


PROJECT_ROOT = find_project_root(Path.cwd())
FREEZE_DIR = PROJECT_ROOT / "outputs" / "01c_mci_survival_cohort_freeze"
OUT_DIR = PROJECT_ROOT / "outputs" / "03a_mci_survival_shared_modeling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "v1"
INPUT_NAME = "mci_survival_primary_cohort_v1.tsv"
INPUT_PATH = FREEZE_DIR / INPUT_NAME
EXPECTED_SHA = "a15e2cb8606f676b3bcfc4d23de72c752779836ce9406b04b01c84394a76e8ad"
EXPECTED = dict(n=401, events=85, censored=316, cols=78)

# ---- THE fixed 26-column output contract, in order ----
FINAL_COLUMNS = [
    "RID", "anchor_date",
    "time_to_event_or_censor_days", "event_indicator",
    "entry_age", "APOE4_COUNT", "APOE4_CARRIER", "ptau217", "log_ptau217",
    "gfap", "log_gfap", "nfl", "log_nfl", "abeta42", "abeta40", "ratio_ab42_ab40",
    "pre_anchor_dementia_n", "qc_dementia_on_or_before_anchor",
    "eligible_core", "eligible_gfap", "eligible_nfl", "eligible_amyloid",
    "eligible_gfap_nfl", "eligible_gfap_amyloid", "eligible_nfl_amyloid", "eligible_full_blood",
]

# Every existing frozen v1 artifact is READ-ONLY. Hashed before/after; asserted unchanged.
PROTECTED = [
    "mci_survival_anchor_cohort_v1.tsv", "mci_survival_followup_cohort_v1.tsv",
    "mci_survival_primary_cohort_v1.tsv", "mci_survival_exclusion_log_v1.tsv",
    "mci_survival_cohort_flow_v1.tsv", "mci_survival_freeze_provenance_v1.json",
    "mci_survival_manual_qc_cases_v1.tsv", "mci_survival_manual_qc_longitudinal_context_v1.tsv",
    "mci_survival_manual_qc_form_v1.tsv", "mci_survival_manual_qc_sampling_summary_v1.tsv",
    "mci_survival_feature_missingness_v1.tsv", "mci_survival_model_complete_case_counts_v1.tsv",
    "mci_survival_feature_timing_v1.tsv", "mci_survival_feature_scenario_counts_v1.tsv",
    "mci_survival_ptau181_platform_availability_v1.tsv", "mci_survival_data_dictionary_v1.tsv",
    "mci_survival_freeze_manifest_v1.json",
]

ASSERTIONS: list[dict] = []


def check(name, ok, detail=""):
    ok = bool(ok)
    ASSERTIONS.append({"assertion": name, "passed": ok, "detail": str(detail)})
    if not ok:
        raise AssertionError(f"ASSERTION FAILED: {name}\n  {detail}")
    print(f"  PASS  {name}" + (f"  [{detail}]" if detail else ""))


def rid_list(rids, limit=15):
    rids = sorted(int(r) for r in rids)
    return f"n={len(rids)} RIDs=[{', '.join(str(r) for r in rids[:limit])}" \
           f"{', ...' if len(rids) > limit else ''}]"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def save_tsv(df, name):
    assert name not in PROTECTED, f"REFUSING to overwrite a protected frozen artifact: {name}"
    p = OUT_DIR / name
    df.to_csv(p, sep="\t", index=False)
    print(f"  saved -> {p.relative_to(PROJECT_ROOT)}   ({df.shape[0]} x {df.shape[1]})")
    return p


HASHES_BEFORE = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}

actual = sha256(INPUT_PATH)
print(f"input : {INPUT_NAME}")
print(f"  expected sha256: {EXPECTED_SHA}")
print(f"  actual   sha256: {actual}")
if actual != EXPECTED_SHA:
    cands = sorted(FREEZE_DIR.glob("mci_survival_primary_cohort_*.tsv"))
    raise SystemExit(f"CHECKSUM MISMATCH. Do not continue silently. Candidates: "
                     f"{[c.name for c in cands]}. Investigate before rebuilding.")
check("input checksum matches the expected frozen-v1 primary cohort", actual == EXPECTED_SHA)
print(f"\npython {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

input : mci_survival_primary_cohort_v1.tsv
  expected sha256: a15e2cb8606f676b3bcfc4d23de72c752779836ce9406b04b01c84394a76e8ad
  actual   sha256: a15e2cb8606f676b3bcfc4d23de72c752779836ce9406b04b01c84394a76e8ad
  PASS  input checksum matches the expected frozen-v1 primary cohort

python 3.14.0 | pandas 3.0.1 | numpy 2.4.3


## 1. Load the frozen primary cohort and verify its documented properties

In [2]:
src = pd.read_csv(INPUT_PATH, sep="\t")
n, ncol = len(src), src.shape[1]
n_ev = int((src["event_indicator"] == 1).sum())
n_cn = int((src["event_indicator"] == 0).sum())

check("input has 401 unique participants", n == 401 and src["RID"].nunique() == 401,
      f"rows={n}, unique={src['RID'].nunique()}")
check("input has 85 events / 316 censored", (n_ev, n_cn) == (85, 316), f"{n_ev}/{n_cn}")
check("input has 78 columns", ncol == 78, f"cols={ncol}")
check("input duration strictly positive", bool((src["time_to_event_or_censor_days"] > 0).all()))
check("input event_indicator in {0,1}, no missing",
      set(src["event_indicator"].dropna().unique()) <= {0, 1} and src["event_indicator"].notna().all())

# ---- confirmed concept -> authoritative column map ----
SRC = dict(RID="RID", anchor_date="anchor_date",
           duration="time_to_event_or_censor_days", event="event_indicator",
           entry_age="entry_age", APOE4_COUNT="APOE4_COUNT", ptau217="ptau217",
           gfap="gfap", nfl="nfl", abeta42="abeta42", abeta40="abeta40",
           ratio="ratio_ab42_ab40",
           pre_anchor_dementia_n="pre_anchor_dementia_n",
           qc_dementia="qc_dementia_on_or_before_anchor")
for concept, col in SRC.items():
    check(f"source column '{col}' ({concept}) exists in the frozen cohort", col in src.columns)

# ---- the two prior-dementia QC fields must be UNAMBIGUOUS ----
dem_like = [c for c in src.columns if "dement" in c.lower() or "pre_anchor" in c.lower()
            or c.startswith("qc_")]
print("prior-dementia / QC candidate columns:", dem_like)
check("'pre_anchor_dementia_n' maps unambiguously (exact frozen column; count of pre-anchor dementia dx)",
      "pre_anchor_dementia_n" in src.columns
      and str(src["pre_anchor_dementia_n"].dtype).startswith("int"))
check("'qc_dementia_on_or_before_anchor' maps unambiguously (exact frozen boolean flag)",
      "qc_dementia_on_or_before_anchor" in src.columns)
print(f"  pre_anchor_dementia_n > 0 : {int((src['pre_anchor_dementia_n'] > 0).sum())} participants")
print(f"  qc_dementia flag == True  : {int(src['qc_dementia_on_or_before_anchor'].astype(bool).sum())} participants")
print(f"\nLoaded frozen primary cohort: {n} x {ncol}  ({n_ev} events, {n_cn} censored)")

  PASS  input has 401 unique participants  [rows=401, unique=401]
  PASS  input has 85 events / 316 censored  [85/316]
  PASS  input has 78 columns  [cols=78]
  PASS  input duration strictly positive
  PASS  input event_indicator in {0,1}, no missing
  PASS  source column 'RID' (RID) exists in the frozen cohort
  PASS  source column 'anchor_date' (anchor_date) exists in the frozen cohort
  PASS  source column 'time_to_event_or_censor_days' (duration) exists in the frozen cohort
  PASS  source column 'event_indicator' (event) exists in the frozen cohort
  PASS  source column 'entry_age' (entry_age) exists in the frozen cohort
  PASS  source column 'APOE4_COUNT' (APOE4_COUNT) exists in the frozen cohort
  PASS  source column 'ptau217' (ptau217) exists in the frozen cohort
  PASS  source column 'gfap' (gfap) exists in the frozen cohort
  PASS  source column 'nfl' (nfl) exists in the frozen cohort
  PASS  source column 'abeta42' (abeta42) exists in the frozen cohort
  PASS  source column '

## 2. Assemble the shared dataset — exactly 26 columns

One row per RID, **all 401 preserved**, sorted by RID.

### Derivations (match Person 2's published analysis)
```python
APOE4_CARRIER   = (APOE4_COUNT >= 1).astype(int)
log_ptau217     = np.log(ptau217)
log_gfap        = np.log(gfap)        # NA preserved where gfap missing
log_nfl         = np.log(nfl)         # NA preserved where nfl missing
ratio_ab42_ab40 = abeta42 / abeta40   # carried from the frozen column (identical by construction)
```

### Eligibility flags (per participant; complete-data membership for each analysis set)
`core := entry_age & APOE4_COUNT & ptau217` (complete for all 401). `amyloid := ratio_ab42_ab40`
present (equivalently both Aβ components valid). Each flag is `core AND the named markers present`,
coded `{0,1}`. They **materialize** each complete-case analysis set so downstream teams can filter
with one column and never re-derive availability.

In [3]:
d = src.sort_values("RID").reset_index(drop=True)
out = pd.DataFrame()

# identification & provenance
out["RID"] = d["RID"]                                 # IDENTIFIER ONLY — never a predictor
out["anchor_date"] = d["anchor_date"]                 # provenance / harmonization only
# outcome (frozen; not recomputed)
out["time_to_event_or_censor_days"] = d["time_to_event_or_censor_days"]
out["event_indicator"] = d["event_indicator"].astype(int)
# core predictors
out["entry_age"] = d["entry_age"]
out["APOE4_COUNT"] = d["APOE4_COUNT"]
out["APOE4_CARRIER"] = (d["APOE4_COUNT"] >= 1).astype(int)
out["ptau217"] = d["ptau217"]
out["log_ptau217"] = np.log(d["ptau217"])
# blood biomarkers: raw + log interleaved
out["gfap"] = d["gfap"]
out["log_gfap"] = np.log(d["gfap"])
out["nfl"] = d["nfl"]
out["log_nfl"] = np.log(d["nfl"])
out["abeta42"] = d["abeta42"]
out["abeta40"] = d["abeta40"]
out["ratio_ab42_ab40"] = d["ratio_ab42_ab40"]         # == abeta42/abeta40 (asserted)
# prior-dementia QC fields (carried from frozen)
out["pre_anchor_dementia_n"] = d["pre_anchor_dementia_n"]
out["qc_dementia_on_or_before_anchor"] = d["qc_dementia_on_or_before_anchor"]

# ---- eligibility flags ----
core = d["entry_age"].notna() & d["APOE4_COUNT"].notna() & d["ptau217"].notna()
g, nf, am = d["gfap"].notna(), d["nfl"].notna(), d["ratio_ab42_ab40"].notna()
ELIG = {
    "eligible_core": core,
    "eligible_gfap": core & g,
    "eligible_nfl": core & nf,
    "eligible_amyloid": core & am,
    "eligible_gfap_nfl": core & g & nf,
    "eligible_gfap_amyloid": core & g & am,
    "eligible_nfl_amyloid": core & nf & am,
    "eligible_full_blood": core & g & nf & am,
}
for name, flag in ELIG.items():
    out[name] = flag.astype(int)

out = out[FINAL_COLUMNS]
print(f"Shared dataset: {out.shape[0]} x {out.shape[1]}")
for i, c in enumerate(out.columns, 1):
    print(f"  {i:2d}  {c:34s} {str(out[c].dtype):8s} missing={int(out[c].isna().sum())}")
out.head()

Shared dataset: 401 x 26
   1  RID                                int64    missing=0
   2  anchor_date                        str      missing=0
   3  time_to_event_or_censor_days       float64  missing=0
   4  event_indicator                    int64    missing=0
   5  entry_age                          float64  missing=0
   6  APOE4_COUNT                        float64  missing=0
   7  APOE4_CARRIER                      int64    missing=0
   8  ptau217                            float64  missing=0
   9  log_ptau217                        float64  missing=0
  10  gfap                               float64  missing=40
  11  log_gfap                           float64  missing=40
  12  nfl                                float64  missing=40
  13  log_nfl                            float64  missing=40
  14  abeta42                            float64  missing=2
  15  abeta40                            float64  missing=2
  16  ratio_ab42_ab40                    float64  missing=2
  17  pre_a

,RID,anchor_date,time_to_event_or_censor_days,event_indicator,entry_age,APOE4_COUNT,APOE4_CARRIER,ptau217,log_ptau217,gfap,log_gfap,nfl,log_nfl,abeta42,abeta40,ratio_ab42_ab40,pre_anchor_dementia_n,qc_dementia_on_or_before_anchor,eligible_core,eligible_gfap,eligible_nfl,eligible_amyloid,eligible_gfap_nfl,eligible_gfap_amyloid,eligible_nfl_amyloid,eligible_full_blood
0,8,2015-10-29,12.0,0,84.50,0.0,0,0.235,-1.448170,289.10,5.666773,55.8,4.021774,7.24,70.00,0.103429,0,False,1,1,1,1,1,1,1,1
1,42,2005-11-10,364.0,1,72.88,0.0,0,0.095,-2.353878,172.10,5.148076,73.3,4.294561,26.85,288.11,0.093194,0,False,1,1,1,1,1,1,1,1
2,69,2024-04-01,490.0,0,72.93,0.0,0,0.249,-1.390302,344.60,5.842384,40.3,3.696351,43.71,488.93,0.089399,0,False,1,1,1,1,1,1,1,1
3,108,2006-02-14,541.0,1,78.28,1.0,1,0.606,-0.500875,257.30,5.550243,33.4,3.508556,26.65,317.70,0.083884,0,False,1,1,1,1,1,1,1,1
4,127,2017-08-23,2840.0,0,70.68,0.0,0,0.189,-1.666008,90.68,4.507337,14.4,2.667228,32.64,296.28,0.110166,0,False,1,1,1,1,1,1,1,1


## 3. Validation

In [4]:
# ---- exact column contract ----
check("output has EXACTLY the 26 specified columns, in the specified order",
      list(out.columns) == FINAL_COLUMNS,
      f"got {list(out.columns)}")

# ---- participant set preserved ----
check("401 rows, one per RID, no duplicates, sorted by RID",
      len(out) == 401 and out["RID"].is_unique and out["RID"].notna().all()
      and out["RID"].is_monotonic_increasing)
check("RID set == frozen primary cohort (no participant added or removed)",
      set(out["RID"]) == set(src["RID"]), rid_list(set(out["RID"]) ^ set(src["RID"])))

# ---- outcome carried faithfully ----
_m = out.merge(src[["RID", "time_to_event_or_censor_days", "event_indicator"]], on="RID",
               suffixes=("", "_s"))
check("duration + event carried byte-faithfully (not recomputed); durations strictly positive",
      bool((_m["time_to_event_or_censor_days"] == _m["time_to_event_or_censor_days_s"]).all())
      and bool((_m["event_indicator"] == _m["event_indicator_s"]).all())
      and bool((out["time_to_event_or_censor_days"] > 0).all()))
check("event counts preserved (85 / 316)",
      int((out["event_indicator"] == 1).sum()) == 85 and int((out["event_indicator"] == 0).sum()) == 316)

# ---- core predictors complete ----
for c in ["entry_age", "APOE4_COUNT", "APOE4_CARRIER", "ptau217", "log_ptau217"]:
    check(f"core predictor '{c}' complete for all 401", out[c].notna().all())

# ---- derivations faithful ----
check("APOE4_CARRIER == (APOE4_COUNT>=1), coded {0,1}",
      bool((out["APOE4_CARRIER"] == (d["APOE4_COUNT"] >= 1).astype(int)).all())
      and set(out["APOE4_CARRIER"].unique()) <= {0, 1})
for lc, rc in [("log_ptau217", "ptau217"), ("log_gfap", "gfap"), ("log_nfl", "nfl")]:
    check(f"{lc} == np.log({rc}); NA exactly where {rc} is NA; no +/-inf",
          bool(np.allclose(out[lc], np.log(d[rc]), equal_nan=True))
          and bool((out[lc].isna() == d[rc].isna()).all())
          and not bool(np.isinf(out[lc].fillna(0)).any()))
check("ratio_ab42_ab40 == abeta42/abeta40 (carried from frozen)",
      bool(np.allclose(out["ratio_ab42_ab40"], d["abeta42"] / d["abeta40"], equal_nan=True)))

# ---- QC fields carried faithfully ----
check("pre_anchor_dementia_n carried byte-faithfully from frozen",
      bool((out["pre_anchor_dementia_n"].values == d["pre_anchor_dementia_n"].values).all()))
check("qc_dementia_on_or_before_anchor carried byte-faithfully from frozen",
      bool((out["qc_dementia_on_or_before_anchor"].astype(bool).values
            == d["qc_dementia_on_or_before_anchor"].astype(bool).values).all()))
check("prior-dementia flag agrees with the count field (flag <=> n>0)",
      bool((out["qc_dementia_on_or_before_anchor"].astype(bool)
            == (out["pre_anchor_dementia_n"] > 0)).all()),
      f"flagged={int(out['qc_dementia_on_or_before_anchor'].astype(bool).sum())}")

# ---- eligibility flags correct ----
EXP_ELIG = {"eligible_core": 401, "eligible_gfap": 361, "eligible_nfl": 361, "eligible_amyloid": 399,
            "eligible_gfap_nfl": 361, "eligible_gfap_amyloid": 359, "eligible_nfl_amyloid": 359,
            "eligible_full_blood": 359}
for name, exp in EXP_ELIG.items():
    got = int(out[name].sum())
    check(f"{name}: {exp} eligible participants (recomputed from nonmissingness)", got == exp, f"got {got}")
    check(f"{name} is coded strictly {{0,1}}", set(out[name].unique()) <= {0, 1})
check("eligibility flags are properly nested (full_blood subset-of gfap_nfl subset-of gfap/nfl subset-of core)",
      bool((out["eligible_full_blood"] <= out["eligible_gfap_nfl"]).all())
      and bool((out["eligible_gfap_nfl"] <= out["eligible_gfap"]).all())
      and bool((out["eligible_gfap"] <= out["eligible_core"]).all()))
check("eligible_core == 1 for all 401 (core predictors complete by construction)",
      bool((out["eligible_core"] == 1).all()))

# ---- NO imputation ----
EXPECT_MISS = {"gfap": 40, "log_gfap": 40, "nfl": 40, "log_nfl": 40,
               "abeta42": 2, "abeta40": 2, "ratio_ab42_ab40": 2}
for c, exp in EXPECT_MISS.items():
    check(f"no imputation: '{c}' has {exp} NA, matching frozen", int(out[c].isna().sum()) == exp)
check("dataset NOT reduced to any complete-case subset (all 401 retained)", len(out) == 401)

# ---- NO scaling: raw carried columns byte-identical to frozen ----
for c in ["entry_age", "APOE4_COUNT", "ptau217", "gfap", "nfl", "abeta42", "abeta40",
          "ratio_ab42_ab40", "pre_anchor_dementia_n"]:
    a = out.set_index("RID")[c].sort_index()
    b = src.set_index("RID")[c].sort_index()
    check(f"no scaling: '{c}' byte-identical to frozen", a.equals(b))

# ---- prohibited content absent ----
check("no cognitive score present",
      not any(any(k in col for k in ["CDRSB", "MMSCORE", "MOCA", "FAQTOTAL", "cdrsb", "mmse",
                                     "moca", "faq"]) for col in out.columns))
check("no CV-fold column present",
      not any("fold" in c.lower() or c.lower().startswith("cv_") for c in out.columns))

  PASS  output has EXACTLY the 26 specified columns, in the specified order  [got ['RID', 'anchor_date', 'time_to_event_or_censor_days', 'event_indicator', 'entry_age', 'APOE4_COUNT', 'APOE4_CARRIER', 'ptau217', 'log_ptau217', 'gfap', 'log_gfap', 'nfl', 'log_nfl', 'abeta42', 'abeta40', 'ratio_ab42_ab40', 'pre_anchor_dementia_n', 'qc_dementia_on_or_before_anchor', 'eligible_core', 'eligible_gfap', 'eligible_nfl', 'eligible_amyloid', 'eligible_gfap_nfl', 'eligible_gfap_amyloid', 'eligible_nfl_amyloid', 'eligible_full_blood']]
  PASS  401 rows, one per RID, no duplicates, sorted by RID
  PASS  RID set == frozen primary cohort (no participant added or removed)  [n=0 RIDs=[]]
  PASS  duration + event carried byte-faithfully (not recomputed); durations strictly positive
  PASS  event counts preserved (85 / 316)
  PASS  core predictor 'entry_age' complete for all 401
  PASS  core predictor 'APOE4_COUNT' complete for all 401
  PASS  core predictor 'APOE4_CARRIER' complete for all 401
  PASS  c

## 4. Analysis-set counts

N / events / censored for each `eligible_*` analysis set, computed straight from the flags. The shared
file is never subset here — this documents the sample cost of each feature combination.

In [5]:
SET_META = {
    "eligible_core": ("entry_age+APOE4_COUNT+log_ptau217", "core primary model (== frozen primary cohort)"),
    "eligible_gfap": ("core+log_gfap", "+ GFAP"),
    "eligible_nfl": ("core+log_nfl", "+ NfL"),
    "eligible_amyloid": ("core+ratio_ab42_ab40", "+ Abeta42/Abeta40 ratio"),
    "eligible_gfap_nfl": ("core+log_gfap+log_nfl", "+ GFAP + NfL"),
    "eligible_gfap_amyloid": ("core+log_gfap+ratio_ab42_ab40", "+ GFAP + amyloid ratio"),
    "eligible_nfl_amyloid": ("core+log_nfl+ratio_ab42_ab40", "+ NfL + amyloid ratio"),
    "eligible_full_blood": ("core+log_gfap+log_nfl+ratio_ab42_ab40",
                            "all blood biomarkers (GFAP+NfL+ratio) complete-case"),
}
rows = []
for flag, (preds, use) in SET_META.items():
    m = out[flag] == 1
    ev = int((out.loc[m, "event_indicator"] == 1).sum())
    npt = int(m.sum())
    npred = 3 + flag.count("_") if flag != "eligible_core" else 3   # rough predictor count
    rows.append(dict(analysis_set=flag, eligibility_flag=flag, required_predictors=preds,
                     n=npt, events=ev, censored=npt - ev, pct_of_401=round(100 * npt / 401, 1),
                     intended_use=use))
counts = pd.DataFrame(rows)

# reconcile flag sums with count rows
for _, r in counts.iterrows():
    check(f"counts row '{r['analysis_set']}' N == its eligibility-flag sum",
          int(r["n"]) == int(out[r["eligibility_flag"]].sum()))
check("core analysis set reproduces the frozen primary cohort (401 / 85)",
      tuple(counts.loc[counts["analysis_set"] == "eligible_core", ["n", "events"]].iloc[0]) == (401, 85))
check("full_blood analysis set = 359 / 83",
      tuple(counts.loc[counts["analysis_set"] == "eligible_full_blood", ["n", "events"]].iloc[0]) == (359, 83))
print(counts[["analysis_set", "required_predictors", "n", "events", "censored", "pct_of_401"]].to_string(index=False))

  PASS  counts row 'eligible_core' N == its eligibility-flag sum
  PASS  counts row 'eligible_gfap' N == its eligibility-flag sum
  PASS  counts row 'eligible_nfl' N == its eligibility-flag sum
  PASS  counts row 'eligible_amyloid' N == its eligibility-flag sum
  PASS  counts row 'eligible_gfap_nfl' N == its eligibility-flag sum
  PASS  counts row 'eligible_gfap_amyloid' N == its eligibility-flag sum
  PASS  counts row 'eligible_nfl_amyloid' N == its eligibility-flag sum
  PASS  counts row 'eligible_full_blood' N == its eligibility-flag sum
  PASS  core analysis set reproduces the frozen primary cohort (401 / 85)
  PASS  full_blood analysis set = 359 / 83
         analysis_set                   required_predictors   n  events  censored  pct_of_401
        eligible_core     entry_age+APOE4_COUNT+log_ptau217 401      85       316       100.0
        eligible_gfap                         core+log_gfap 361      85       276        90.0
         eligible_nfl                          core+lo

## 5. Data dictionary — one row per shared-dataset column (26)

In [6]:
def R(col, label, unit, allowed, source, deriv, role, transform, note):
    return dict(output_column=col, label=label, unit=unit, allowed_values=allowed, source=source,
                derivation=deriv, role=role, transformation_applied=transform, notes=note)


FZ = "frozen mci_survival_primary_cohort_v1.tsv"
DD = [
 R("RID", "Participant ID", "integer id", "positive integer", FZ, "carried", "IDENTIFIER", "none",
   "IDENTIFIER ONLY — never a predictor"),
 R("anchor_date", "Plasma anchor date", "date", "YYYY-MM-DD", FZ, "carried", "PROVENANCE", "none",
   "provenance / NACC harmonization only"),
 R("time_to_event_or_censor_days", "Follow-up duration", "days", "strictly positive", FZ,
   "carried (not recomputed)", "OUTCOME (duration)", "none", "use with event_indicator"),
 R("event_indicator", "Incident all-cause dementia", "0/1", "1=dementia,0=censored", FZ,
   "carried (not recomputed)", "OUTCOME (event)", "none", "0 = censored, NOT 'stable'"),
 R("entry_age", "Age at study entry", "years", "positive float", FZ, "carried", "PRIMARY PREDICTOR",
   "none", "UNDATED; age at entry, not at anchor"),
 R("APOE4_COUNT", "APOE e4 allele count", "alleles", "0/1/2", FZ, "carried", "PRIMARY PREDICTOR",
   "none", "complete in the primary cohort"),
 R("APOE4_CARRIER", "APOE e4 carrier", "0/1", "0/1", "derived", "(APOE4_COUNT>=1).astype(int)",
   "PRIMARY PREDICTOR (derived)", "binarization", "matches Person 2's sensitivity model"),
 R("ptau217", "Plasma p-tau217 (raw)", "pg/mL", "positive float", FZ, "carried", "PRIMARY PREDICTOR",
   "none", "raw Fujirebio; complete in the primary cohort"),
 R("log_ptau217", "Natural log p-tau217", "log pg/mL", "float", "derived", "np.log(ptau217)",
   "PRIMARY PREDICTOR (derived)", "natural log", "transform used in Person 2's Model 1"),
 R("gfap", "Plasma GFAP (raw)", "pg/mL", "positive float", FZ, "carried", "SECONDARY PREDICTOR",
   "none", "Quanterix; missing for the same 40 as nfl"),
 R("log_gfap", "Natural log GFAP", "log pg/mL", "float", "derived", "np.log(gfap)",
   "SECONDARY PREDICTOR (derived)", "natural log", "NA preserved where gfap missing"),
 R("nfl", "Plasma NfL (raw)", "pg/mL", "positive float", FZ, "carried", "SECONDARY PREDICTOR",
   "none", "Quanterix; missing for the same 40 as gfap"),
 R("log_nfl", "Natural log NfL", "log pg/mL", "float", "derived", "np.log(nfl)",
   "SECONDARY PREDICTOR (derived)", "natural log", "NA preserved where nfl missing"),
 R("abeta42", "Plasma Abeta-42 (raw)", "pg/mL", "positive float", FZ, "carried", "SECONDARY PREDICTOR",
   "none", "Fujirebio; ratio numerator (preserved)"),
 R("abeta40", "Plasma Abeta-40 (raw)", "pg/mL", "positive float", FZ, "carried", "SECONDARY PREDICTOR",
   "none", "Fujirebio; ratio denominator (preserved)"),
 R("ratio_ab42_ab40", "Abeta-42/Abeta-40 ratio", "ratio", "positive float", FZ,
   "abeta42/abeta40 (== frozen col)", "SECONDARY PREDICTOR", "none",
   "same raw row for both components; NA where either missing"),
 R("pre_anchor_dementia_n", "Count of dementia dx before the anchor", "count", ">=0", FZ, "carried",
   "QC FLAG", "none", "prior-dementia QC; NOT an exclusion; adjudication pending"),
 R("qc_dementia_on_or_before_anchor", "Any dementia dx on/before the anchor", "bool", "True/False",
   FZ, "carried", "QC FLAG", "none",
   "prior-dementia QC flag; NOT an automatic exclusion; adjudication pending"),
 R("eligible_core", "Complete for core (age+APOE4+ptau217)", "0/1", "0/1", "derived",
   "age & APOE4 & ptau217 present", "ELIGIBILITY FLAG", "none", "== 1 for all 401"),
 R("eligible_gfap", "Complete for core + GFAP", "0/1", "0/1", "derived", "core & gfap present",
   "ELIGIBILITY FLAG", "none", "361 eligible"),
 R("eligible_nfl", "Complete for core + NfL", "0/1", "0/1", "derived", "core & nfl present",
   "ELIGIBILITY FLAG", "none", "361 eligible"),
 R("eligible_amyloid", "Complete for core + amyloid ratio", "0/1", "0/1", "derived",
   "core & ratio_ab42_ab40 present", "ELIGIBILITY FLAG", "none", "399 eligible"),
 R("eligible_gfap_nfl", "Complete for core + GFAP + NfL", "0/1", "0/1", "derived",
   "core & gfap & nfl present", "ELIGIBILITY FLAG", "none", "361 eligible"),
 R("eligible_gfap_amyloid", "Complete for core + GFAP + amyloid", "0/1", "0/1", "derived",
   "core & gfap & ratio present", "ELIGIBILITY FLAG", "none", "359 eligible"),
 R("eligible_nfl_amyloid", "Complete for core + NfL + amyloid", "0/1", "0/1", "derived",
   "core & nfl & ratio present", "ELIGIBILITY FLAG", "none", "359 eligible"),
 R("eligible_full_blood", "Complete for core + GFAP + NfL + amyloid", "0/1", "0/1", "derived",
   "core & gfap & nfl & ratio present", "ELIGIBILITY FLAG", "none", "359 eligible (== full blood panel)"),
]
dd = pd.DataFrame(DD)
dd["missing_in_401"] = dd["output_column"].map(lambda c: int(out[c].isna().sum()))
dd["data_type"] = dd["output_column"].map(lambda c: str(out[c].dtype))
dd = dd[["output_column", "label", "data_type", "unit", "allowed_values", "source", "derivation",
         "role", "missing_in_401", "transformation_applied", "notes"]]

check("data dictionary covers EVERY column, in the exact output order",
      list(dd["output_column"]) == list(out.columns) == FINAL_COLUMNS)
check("dictionary missing-counts match the dataset",
      bool((dd.set_index("output_column")["missing_in_401"].reindex(out.columns).values
            == out.isna().sum().values).all()))
print(f"Data dictionary: {len(dd)} rows")
print(dd[["output_column", "role", "missing_in_401", "transformation_applied"]].to_string(index=False))

  PASS  data dictionary covers EVERY column, in the exact output order
  PASS  dictionary missing-counts match the dataset
Data dictionary: 26 rows
                  output_column                          role  missing_in_401 transformation_applied
                            RID                    IDENTIFIER               0                   none
                    anchor_date                    PROVENANCE               0                   none
   time_to_event_or_censor_days            OUTCOME (duration)               0                   none
                event_indicator               OUTCOME (event)               0                   none
                      entry_age             PRIMARY PREDICTOR               0                   none
                    APOE4_COUNT             PRIMARY PREDICTOR               0                   none
                  APOE4_CARRIER   PRIMARY PREDICTOR (derived)               0           binarization
                        ptau217             

## 6. Write outputs + manifest; verify frozen v1 artifacts byte-identical

In [7]:
def git(*a):
    try:
        return subprocess.run(["git", *a], cwd=PROJECT_ROOT, capture_output=True, text=True,
                              check=True).stdout.strip()
    except Exception:
        return None


ds_path = save_tsv(out, f"mci_survival_shared_modeling_dataset_{VERSION}.tsv")
dd_path = save_tsv(dd, f"mci_survival_shared_modeling_data_dictionary_{VERSION}.tsv")
ct_path = save_tsv(counts, f"mci_survival_analysis_set_counts_{VERSION}.tsv")
OUTPUT_FILES = {ds_path.name: (ds_path, out), dd_path.name: (dd_path, dd), ct_path.name: (ct_path, counts)}

manifest = {
    "manifest_schema_version": "1.0.0",
    "dataset": "mci_survival_shared_modeling_dataset", "version": VERSION,
    "purpose": ("shared analysis-ready survival-modeling dataset for feature-combination analysis, "
                "alternative survival-model comparison, centralized evaluation, and NACC "
                "harmonization; all 401 primary-cohort participants preserved; exactly 26 columns"),
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "created_utc_note": "the only non-deterministic field; excluded from byte-identity comparisons",
    "self_hash_policy": "this manifest excludes its own SHA-256; verify externally with shasum",
    "git": {"commit": git("rev-parse", "HEAD"), "branch": git("rev-parse", "--abbrev-ref", "HEAD"),
            "dirty": bool(git("status", "--porcelain"))},
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "numpy": np.__version__, "platform": platform.platform()},
    "input": {"path": f"outputs/01c_mci_survival_cohort_freeze/{INPUT_NAME}", "sha256": actual,
              "expected_sha256": EXPECTED_SHA, "checksum_verified": True,
              "n_participants": n, "n_events": n_ev, "n_censored": n_cn, "n_columns": ncol},
    "final_columns": FINAL_COLUMNS,
    "derivations": {"APOE4_CARRIER": "(APOE4_COUNT >= 1).astype(int)",
                    "log_ptau217": "np.log(ptau217)", "log_gfap": "np.log(gfap)",
                    "log_nfl": "np.log(nfl)",
                    "ratio_ab42_ab40": "abeta42 / abeta40 (carried from frozen; identical)"},
    "eligibility_definitions": {
        "core": "entry_age & APOE4_COUNT & ptau217 all present (complete for all 401)",
        "amyloid": "ratio_ab42_ab40 present (both Abeta components valid)",
        **{k: f"core AND {k.replace('eligible_', '').replace('_', ' + ')} present" for k in ELIG}},
    "policy": {"participants_preserved": 401, "imputation": "none", "scaling": "none",
               "feature_selection": "none", "model_fitting": "none", "cognition_added": False,
               "cv_folds_added": False, "reduced_to_complete_case": False,
               "raw_biomarkers_carried_byte_identical": True,
               "prior_dementia_participants_retained": True},
    "column_roles": {r["output_column"]: r["role"] for r in DD},
    "analysis_set_counts": counts.to_dict(orient="records"),
    "outputs": [{"path": f"outputs/03a_mci_survival_shared_modeling/{name}", "sha256": sha256(p),
                 "n_rows": int(df.shape[0]), "n_cols": int(df.shape[1])}
                for name, (p, df) in OUTPUT_FILES.items()],
    "protected_frozen_artifacts_unchanged": {f: HASHES_BEFORE[f] for f in PROTECTED},
    "report": "reports/mci_survival_build4_shared_modeling_dataset_v1.md",
    "notebook": "notebooks/03a_mci_survival_shared_modeling_dataset.ipynb",
}
MPATH = OUT_DIR / f"mci_survival_shared_modeling_manifest_{VERSION}.json"
MPATH.write_text(json.dumps(manifest, indent=2, default=str))
print(f"  saved -> {MPATH.relative_to(PROJECT_ROOT)}")

HASHES_AFTER = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}
drift = [f for f in PROTECTED if HASHES_BEFORE[f] != HASHES_AFTER[f]]
check("ALL existing frozen v1 artifacts BYTE-IDENTICAL after this build",
      not drift, f"drifted: {drift}" if drift else f"{len(PROTECTED)} unchanged")
check("all outputs written under outputs/03a_mci_survival_shared_modeling/",
      all((OUT_DIR / x).exists() for x in [ds_path.name, dd_path.name, ct_path.name, MPATH.name]))
print("\nOutput hashes:")
for name, (p, _) in OUTPUT_FILES.items():
    print(f"  {name:52s} {sha256(p)[:16]}...")

  saved -> outputs/03a_mci_survival_shared_modeling/mci_survival_shared_modeling_dataset_v1.tsv   (401 x 26)
  saved -> outputs/03a_mci_survival_shared_modeling/mci_survival_shared_modeling_data_dictionary_v1.tsv   (26 x 11)
  saved -> outputs/03a_mci_survival_shared_modeling/mci_survival_analysis_set_counts_v1.tsv   (8 x 8)
  saved -> outputs/03a_mci_survival_shared_modeling/mci_survival_shared_modeling_manifest_v1.json
  PASS  ALL existing frozen v1 artifacts BYTE-IDENTICAL after this build  [17 unchanged]
  PASS  all outputs written under outputs/03a_mci_survival_shared_modeling/

Output hashes:
  mci_survival_shared_modeling_dataset_v1.tsv          521120e76418da54...
  mci_survival_shared_modeling_data_dictionary_v1.tsv  9ff1df31d93c40bd...
  mci_survival_analysis_set_counts_v1.tsv              da724882a09e1d04...


## 7. Summary

In [8]:
print("=" * 90)
print("SHARED SURVIVAL-MODELING DATASET (03a, v1)")
print("=" * 90)
print(f"Assertions : {sum(a['passed'] for a in ASSERTIONS)} passed / "
      f"{sum(not a['passed'] for a in ASSERTIONS)} failed  (of {len(ASSERTIONS)})")
print(f"Shared dataset : {out.shape[0]} participants x {out.shape[1]} columns "
      f"({int((out['event_indicator']==1).sum())} events, {int((out['event_indicator']==0).sum())} censored)")
print("  ALL 401 preserved; optional biomarkers NA where missing; 26-column contract satisfied")
print()
print("Eligibility flags (per-participant analysis-set membership):")
for _, r in counts.iterrows():
    print(f"  {r['analysis_set']:24s} N={int(r['n']):>4d}  events={int(r['events']):<3d} {r['intended_use']}")
print()
print("Prior-dementia QC carried:", int(out['qc_dementia_on_or_before_anchor'].astype(bool).sum()),
      "flagged (adjudication pending; NOT excluded)")
print()
print("Load: outputs/03a_mci_survival_shared_modeling/mci_survival_shared_modeling_dataset_v1.tsv")
print("Frozen v1 unchanged. No imputation, scaling, feature selection, models, cognition, or CV folds.")
print("=" * 90)

SHARED SURVIVAL-MODELING DATASET (03a, v1)
Assertions : 91 passed / 0 failed  (of 91)
Shared dataset : 401 participants x 26 columns (85 events, 316 censored)
  ALL 401 preserved; optional biomarkers NA where missing; 26-column contract satisfied

Eligibility flags (per-participant analysis-set membership):
  eligible_core            N= 401  events=85  core primary model (== frozen primary cohort)
  eligible_gfap            N= 361  events=85  + GFAP
  eligible_nfl             N= 361  events=85  + NfL
  eligible_amyloid         N= 399  events=83  + Abeta42/Abeta40 ratio
  eligible_gfap_nfl        N= 361  events=85  + GFAP + NfL
  eligible_gfap_amyloid    N= 359  events=83  + GFAP + amyloid ratio
  eligible_nfl_amyloid     N= 359  events=83  + NfL + amyloid ratio
  eligible_full_blood      N= 359  events=83  all blood biomarkers (GFAP+NfL+ratio) complete-case

Prior-dementia QC carried: 4 flagged (adjudication pending; NOT excluded)

Load: outputs/03a_mci_survival_shared_modeling/mci_sur